In [0]:
# Libraries and functions 
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number
from pyspark.sql.functions import current_date
from pyspark.sql.functions import col, count, sum, avg, max, min, to_date


In [0]:
%run /Workspace/Users/fernandodamsouza@gmail.com/cdc-databricks/notebooks/utils

In [0]:
# Parameters

dbutils.widgets.text("catalog", "main_cdc", "Catalog")
dbutils.widgets.text("data_source", "sales_cdc_big", "Data Source")

In [0]:
# Get widget values
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
# Paths
raw_path = f"/Volumes/{catalog}/default/raw/"
base_path = f'/Volumes/{catalog}/default/raw/{data_source}.csv'
checkpoint_path_bronze = f"/Volumes/{catalog}/default/checkpoints/bronze_sales"
checkpoint_path_silver = f"/Volumes/{catalog}/default/checkpoints/silver_sales"
checkpoint_path_gold = f"/Volumes/{catalog}/default/checkpoints/gold_sales"
schema_path_bronze = f"/Volumes/{catalog}/default/schema/bronze_sales"
schema_path_silver = f"/Volumes/{catalog}/default/schema/silver_sales"
schema_path_gold = f"/Volumes/{catalog}/default/schema/gold_sales"
full_table_bronze = f"{catalog}.{bronze_schema}.{table_bronze}"
full_table_silver = f"{catalog}.{silver_schema}.{table_silver}"

### BRONZE PROCESSING

In [0]:
# Cria tabela com CDF (se não existir)
table_bronze = spark.sql(f"""
CREATE TABLE IF NOT EXISTS full_table_bronze (
  id INT,
  cliente_id INT,
  produto_id INT,
  valor DOUBLE,
  operation STRING,
  timestamp TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
  delta.enableChangeDataFeed = true
)
""")

In [0]:
spark.sql(f"""
ALTER TABLE full_table_bronze
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

In [0]:
# Auto Loader
df_stream_bronze = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("delimiter", ",") \
    .option("delimiter", ",") \
    .option("header", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("cloudFiles.schemaEvolutionMode", "rescue") \
    .option("header", "true") \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option("cloudFiles.schemaLocation", schema_path_bronze) \
    .option("cloudFiles.includeExistingFiles", "true") \
    .load(raw_path)

# Escrita Bronze (CORRETO)
query = df_stream_bronze.writeStream \
    .format("delta") \
    .option("checkpointLocation", checkpoint_path_bronze) \
    .trigger(availableNow=True) \
    .toTable(full_table_bronze)

query.awaitTermination()

### SILVER PROCESSING

In [0]:
# Cria tabela com CDF (se não existir)
table_silver = spark.sql(f"""
CREATE TABLE IF NOT EXISTS full_table_silver (
  id INT,
  cliente_id INT,
  produto_id INT,
  valor DOUBLE,
  event_timestamp TIMESTAMP,
  stringcolumntest STRING
)
USING DELTA
TBLPROPERTIES (
  delta.enableChangeDataFeed = true
)
""")

In [0]:
# MERGE
def upsert_to_silver(df_batch, batch_id):

    if df_batch.count() == 0:
        return

    # =============================
    # LIMPEZA E HIGIENIZAÇÃO
    # =============================

    df_clean = (
        df_batch
        .withColumn("cliente_id", trim(col("cliente_id")))
        .withColumn("produto_id", trim(col("produto_id")))
        
        # rename + cast     
        .withColumn("valor", col("valor").cast("double"))
               
        # regra de negócio inicial
        .withColumn("valor", when(col("valor") < 0, None).otherwise(col("valor")))
        
        # nova coluna padronizada
        .withColumnRenamed("TesteNewColumn", "stringcolumntest")
    )

    # =============================
    # CDC - DEDUPLICAÇÃO
    # =============================

    window_spec = Window.partitionBy("id").orderBy(col("event_timestamp").desc())

    df_dedup = (
        df_clean
        .withColumn("rn", row_number().over(window_spec))
        .filter("rn = 1")
        .drop("rn")
    )

    # =============================
    # CONTROLE DE SCHEMA
    # =============================

    required_columns = [
        "id", "cliente_id", "produto_id",
        "valor", "event_timestamp",
        "operation", "stringcolumntest"
    ]

    for col_name in required_columns:
        if col_name not in df_dedup.columns:
            df_dedup = df_dedup.withColumn(col_name, lit(None))

    df_dedup = df_dedup.select(*required_columns)

    # =============================
    # MERGE
    # =============================

    target = DeltaTable.forName(spark, full_table_silver)

    (
        target.alias("t")
        .merge(df_dedup.alias("s"), "t.id = s.id")

        # DELETE
        .whenMatchedDelete(condition="s.operation = 'DELETE'")

        # UPDATE
        .whenMatchedUpdate(
            condition="s.operation IN ('UPDATE','INSERT')",
            set={
                "cliente_id": "coalesce(s.cliente_id, t.cliente_id)",
                "produto_id": "coalesce(s.produto_id, t.produto_id)",
                "valor": "coalesce(s.valor, t.valor)",
                "event_timestamp": "coalesce(s.event_timestamp, t.event_timestamp)",
                "stringcolumntest": "coalesce(s.stringcolumntest, t.stringcolumntest)"
            }
        )

        # INSERT
        .whenNotMatchedInsert(
            condition="s.operation = 'INSERT'",
            values={
                "id": "s.id",
                "cliente_id": "s.cliente_id",
                "produto_id": "s.produto_id",
                "valor": "s.valor",
                "event_timestamp": "s.event_timestamp",
                "stringcolumntest": "s.stringcolumntest"
            }
        )
        .execute()
    )


In [0]:
# Stream Silver
silver_stream = spark.readStream.table(full_table_bronze)


### GOLD PROCESSING

In [0]:
# Cria variável com o nome da tabela
full_table_gold = f"{catalog}.{gold_schema}.{table_gold}"

df_gold = silver_stream.groupBy(
    col("cliente_id"),
    to_date(col("timestamp")).alias("data_ref")
).agg(
    count("*").alias("total_compras"),
    sum("valor").alias("faturamento_total"),
    avg("valor").alias("ticket_medio"),
    max("valor").alias("maior_compra"),
    min("valor").alias("menor_compra")
)


In [0]:
df_gold = df_gold.withColumn(
    "categoria_ticket",
    when(col("ticket_medio") < 100, "baixo")
    .when(col("ticket_medio") < 500, "medio")
    .otherwise("alto")
)

In [0]:
# Write Stream Gold
df_gold_write_stream = df_gold.writeStream \
    .outputMode("complete") \
    .format("delta") \
    .option("checkpointLocation", checkpoint_path_gold) \
    .partitionBy("data_ref") \
    .trigger(availableNow=True) \
    .table(full_table_gold)

In [0]:
display(spark.table(full_table_gold))